#### Setup environment and libraries

In [2]:
import os

In [3]:
# For Colab environment
#from google.colab import drive
#drive.mount('/content/drive')


In [4]:
# Change directory
# folder_path = "/content/drive/My Drive/Web Analytics/Final project scraped data/"
folder_path =r"C:\Users\adeli\Documents 4-Q1\Web Analytics\Final Project GitHub\web-analytics-project-repo"
os.chdir(folder_path)

In [5]:
# Check
current_dir = os.getcwd()
print(current_dir)

C:\Users\adeli\Documents 4-Q1\Web Analytics\Final Project GitHub\web-analytics-project-repo


In [6]:
# !ls
# !dir

In [7]:
# Purpose of the notebook: parse json schema returned by the Linkedin scraper
# OBS: not finished to parse:
# Libraries to use
# Standard libraries
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)   # no column truncation
pd.set_option("display.max_rows", None)      # no row truncation
pd.set_option("display.width", None)         # no line-wrapping
pd.set_option("display.max_colwidth", None)
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
# Specific libraries
import json
from pathlib import Path
import re

#### Eliminate duplicate offers

In [ ]:
# ELIMINATION OF DUPLICATE OFFERS CODE
# Duplicates in the Linkedin offers is due to offers showing in locations close to the city scraped.
all_lists = []
files = os.listdir(folder_path)

for f in files:
    if f.endswith(".json"):
        with open(f, "r") as fp:
            data = json.load(fp)
            all_lists.append(data)

merged = [job for lst in all_lists for job in lst]
print(len(merged))

def remove_duplicates(jobs):
    seen = set()
    unique_jobs = []

    for job in jobs:
        url = job.get("url")
        if url and url not in seen:
            seen.add(url)
            unique_jobs.append(job)

    return unique_jobs

# Call the function:
deduped_jobs = remove_duplicates(merged)

# Save the job offers with no duplicates into a json file, for future use.
data = deduped_jobs
with open("linkedin_jobs.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

#### Load and parse the data after de-duplicate

In [17]:
# LOADING AND PARSING TOTAL JSON DATA
# To start from .json point
# Recover file name
file_name = "linkedin_no_duplicated_jobs.json"
path = Path(file_name)

with path.open("r", encoding="utf-8") as f:
    data = json.load(f)               # expects a list[dict]

# If it's a single dict, make it a list
if isinstance(data, dict):
    data = [data]

# Flatten nested fields (company.name -> company_name)
df = pd.json_normalize(data, sep=".")

# Create offer ID using the index (checked that it is unique)
df["offer_id"] = df.index

In [18]:
# EXTRACT CRITERIA INFORMATION INTO COLUMNS
# Explode the criteria column: extract the content of the dict in four columns
df["criteria_dict"] = df["criteria"].apply(
    lambda lst: {d["name"]: d["value"] for d in lst}
)
criteria_expanded = df["criteria_dict"].apply(pd.Series)

df = pd.concat([df, criteria_expanded], axis=1)
df = df.drop(columns=["criteria", "criteria_dict"])

# RENAME AND REORDER COLUMNS
# Rename columns to have the same naming criteria
df = df.rename(columns={
    "company.name": "company_name",
    "company.url": "company_url",
    "Seniority level": "seniority_level",
    "Employment type": "job_type",
    "Job function": "job_function",
    "Industries": "industry"
})

# Reorder the cols as we want them
"""ordered = ['offer_id', "title", "company_name", "location", "salary",
           "description", "url", 'criteria', "seniority_level",'job_type',
           'job_function','industry','searched_position', 'searched_location',
           'applications', 'company_url']
df=df[ordered]"""

# PARSE SALARY INFORMATION
# Ensure None are numpy NaN
df["salary"] = df["salary"].replace([None, "None", ""], np.nan)

# User regular expressions to extract the information we want from the current salary column
# We define first dicts that contain the unit patterns we want to detect
_unit_patterns = [
    (r"\b(per|an|a)\s+hour\b|/hour|/hr|\bhr\b|\bhour\b", "hour"),
    (r"\b(per|an|a)\s+year\b|/year|\byr\b|\byear\b|\bannum\b", "year"),
    (r"\b(per|an|a)\s+month\b|/month|\bmo\b|\bmonth\b", "month"),
    (r"\b(per|an|a)\s+week\b|/week|\bwk\b|\bweek\b", "week"),
    (r"\bdaily\b|\bper\s+day\b|/day|\bday\b", "day"),
]
# Same for the digits
_num_pattern = re.compile(
    r"\$?\s*([0-9]{1,3}(?:,[0-9]{3})*(?:\.[0-9]+)?|[0-9]+(?:\.[0-9]+)?)"
)

# Function to detect the unit of time the salary is given in
def _detect_unit(text_lower):
    for pattern, unit in _unit_patterns:
        if re.search(pattern, text_lower):
            return unit
    return pd.NA

# Function to parse the content of the current salary column
def parse_salary_cell(s):
    """
    Returns (min_val, max_val, unit) where min/max are floats (NaN if not found),
    and unit in {'hour','month','year','week', NA}.
    """
    if pd.isna(s):
        return np.nan, np.nan, pd.NA
    # Remove from the string whitespaces and convert to lower case
    txt = str(s).strip()
    txt_lower = txt.lower()

    # Find all numeric parts in the string
    nums = [float(n.replace(",", "")) for n in _num_pattern.findall(txt)]
    if not nums:
        return np.nan, np.nan, _detect_unit(txt_lower)

    # If we recover one number, assign to both min and max
    # If we recover two numbers, assing as corresponds to min and max
    if len(nums) >= 2:
        low, high = sorted(nums[:2])
    else:
        low = high = nums[0]

    unit = _detect_unit(txt_lower)
    return low, high, unit

# Apply to the column and create the new columns
parsed = df["salary"].apply(parse_salary_cell)
df[["salary_min", "salary_max", "salary_unit"]] = pd.DataFrame(parsed.tolist(), index=df.index)

In [22]:
# PARSE LOCATION INFORMATION
# Extract from location the city and the US state
# Create the mapping dictionary from text name of state to state code. OBS: not needed for Linkedin, was needed in Indeed
state_map = {
    "california": "CA"
}
# Initialize columns
df["city"] = pd.NA
df["state"] = pd.NA

# Split location first
splits = df["location"].str.split(",")

# Compute len per row
len_location = splits.str.len()
# In first Linkedin json only get location with 2 splits, but include the rest of the code in case there are entries
# with only one split as in Indeed data
# If len_location == 1
mask1 = len_location == 1
df.loc[mask1, "city"] = pd.NA
df.loc[mask1, "state"] = splits.str[-1]
# Over this recovered state name, apply the mapping to encode the state code
df["state"] = (
    df["state"]
    .str.lower()
    .replace(state_map, regex=False)
)
# If len_location > 1
mask2 = len_location > 1
df.loc[mask2, "city"] = splits.str[-2]
df.loc[mask2, "state"] = splits.str[-1]
# Need to remove the postal code
df["state_code"] = df["state"].str.split().str[0]

In [25]:
# CODE TO TREAT ENTRIES IN WHICH 'salary' DOES NOT CONTAIN SALARY INFORMATION
# Try to capture the entries where salary is not None or contains a $ -> these are entries where there is
# another information in salary. May be usueful.
# OBS: had to add euros as there is one entry in that currency.
# Then, keeep the entry as if the currency was $ (assume a 1:1 exchange rate, as it is only one entry)
df_no_dollar = df[
    df["salary"].apply(lambda x: isinstance(x, str)) &
    ~df["salary"].str.contains(r"[€$]", na=False)
].copy()

df_no_dollar = df_no_dollar[["offer_id", "salary"]].rename(
    columns={"salary": "useful_type"}
)
len_df_no_dollar = len(df_no_dollar)
print(f"Number of entries with no salary information in 'salary' column: {len_df_no_dollar}")



Number of entries with no salary information in 'salary' column: 0


In [27]:
# CELL OF CODE NOT NEEDED FOR LINKEDIN DATA
# as all entries contain salary information in 'salary' column
# If len_df_no_dollar is not 0
if len_df_no_dollar != 0:
    df = df.merge(df_no_dollar, on="offer_id", how="left")
    # Now extract information from job_type: eliminate 'Job type '
    df["job_type"] = df["job_type"].str.replace("Job type ", "", regex=False)

    # Compare job_type and useful_type (information previously extracted from 'salary'
    # when its content is not in fact a salary). Create a col to store the comparison.
    df["job_vs_useful_match"] = None

    mask = df["useful_type"].notna()
    df.loc[mask, "job_vs_useful_match"] = (
        df.loc[mask, "job_type"] == df.loc[mask, "useful_type"]
    )
    # Print the values of 'job_vs_useful_match'. If all True and the same number than
    # "Number of entries with no salary information in 'salary' column: ", the information in
    # useful_type is the same than the one contained in "job_type".
    counts = df.value_counts("job_vs_useful_match")[True]

    # Print the values of 'job_vs_useful_match'. If all True and the same number than
    # "Number of entries with no salary information in 'salary' column: ", the information in
    # useful_type is the same than the one contained in "job_type".
    counts = df.value_counts("job_vs_useful_match")[True]
    print("Number of entries with the same content in useful_type than in job_type: ", counts)
    print("If same number, columns useful_type and job_vs_useful_match can be dropped from df")

In [28]:
# SAVE URL to OFFER_ID AND DROP
# Before dropping 'url', store csv with offer_id, url in case it is useful in the future.
df[["offer_id", "url"]].to_csv("offer_id_url_Linkedin.csv", index=False)

# DROP, REORDER AND STORE DF into CSV
# Columns determined to be non-useful are dropped: url, company_url, pay
df.drop(columns=["url", "company_url", "state", "location"], inplace=True)

# Reorder the df into our preferred order
print("Num cols before reordering: ", len(df.columns))
ordered = ['offer_id', "title", "company_name", 'city', 'state_code','seniority_level',
           'job_type', 'job_function', 'industry',
           "description", 'salary_min', 'salary_max', 'salary_unit',"salary",
           'searched_position', 'searched_location', 'applications']
print("Num cols after reordering: ", len(ordered))

df = df[ordered]

Num cols before reordering:  17
Num cols after reordering:  17
Data has been saved to file: jobs_linkedin_solvang.csv


In [36]:
# To save
file_name= "jobs_linkedin_parsed.csv"
df.to_csv(file_name, index=False)
print(f"Data has been saved to file: {file_name}")

Data has been saved to file: jobs_linkedin_parsed.csv


#### Computation of embeddings for 'description'

##### Downloads

In [ ]:
!pip install langid
!pip install contractions
!pip install lxml
!pip install swifter
!pip install compress-fasttext

from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag

import re
import contractions
import gensim
from gensim.models.phrases import Phrases
from gensim.corpora import Dictionary
from scipy.sparse import csr_matrix
import swifter

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

##### Preprocessing/ Cleaning of data

In [ ]:
# Define the text preprocessing function
def prepare_data(text, wnl, stopwords_list):
        # Remove HTML tags
        soup = BeautifulSoup(text, "lxml")
        text = soup.get_text()

        # Remove URLs
        text = re.sub(r'https?://\S+|www\.\S+', '', text)

        # Expand contractions
        wrangled_text = contractions.fix(text)

        # Tokenization
        linkedin_tokens = word_tokenize(wrangled_text)

        # POS tagging
        pos_tags = pos_tag(linkedin_tokens)

        # Initialize the lemmatizer
        lemmatizer = wnl

        lemmatized_tokens = []

        for word, pos in pos_tags:
            # Keep only nouns, verbs, adjectives
            if pos.startswith('NN'):
                pos = 'n'  # Noun
            elif pos.startswith('VB'):
                pos = 'v'  # Verb
            elif pos.startswith('JJ'):
                pos = 'a'  # Adjective
            else:
                # Ignore other POS
                continue

            # Lemmatize the word using the appropriate POS tag
            lemmatized_token = lemmatizer.lemmatize(word, pos)
            lemmatized_tokens.append(lemmatized_token)

        # Lowercasing and filtering non-alphanumeric tokens and digits.
        linkedin_tokens_filtered = [token.lower() for token in lemmatized_tokens if token.isalnum() and not token.isdigit()]

        # Remove stopwords
        exclude_lemmas = ['doe', 'wa']
        clean_linkedin = [token for token in linkedin_tokens_filtered if token not in stopwords_list and token not in exclude_lemmas]

        # Return the cleaned list of tokens
        return clean_linkedin
stopwords_en = set(stopwords.words('english'))
wnl = WordNetLemmatizer()
custom_stopwords = {'extra', 'without', 'enough', 'would', 'could', 'lot', 'thank', 'thanks', 'also', 'well', 'much', 'many', 'another', 'others', 'though', 'instead'}
combined_stopwords = stopwords_en.union(custom_stopwords)



Apply the method `prepare_data()` to each entry of the `description` column of the dataframe. To improve the processing speed, we use the `swifter` module to parallelize the proccess of several entries across the CPU.

In [ ]:
import swifter

df['nltk_lemmas'] = df['description'].swifter.apply(prepare_data, args=(wnl, combined_stopwords))
# Check results
print('============= Original text =============')
print(df.iloc[0]['description'], '\n')
print('============= Cleaned text =============')
print(df.iloc[0]['nltk_lemmas'], '\n')
# save progress
df.to_csv("linkedin_description_clean.csv", index = False, encoding = 'utf-8')

##### Text Vectorizaction

In [ ]:
# starting checkpoint
df = pd.read_csv('linkedin_description_clean.csv')
df['nltk_lemmas'] = df['nltk_lemmas'].apply(ast.literal_eval)
df['nltk_lemmas'].head(5)

Generate n-grams and update the `corpus`.

In [ ]:
for mc in [5, 10, 20, 30]:
    for th in [10, 20, 30, 40]:
        phrases = Phrases(df['nltk_lemmas'].tolist(), min_count=mc, threshold=th)
        print(f"min_count={mc}, threshold={th} → n_phrases={len(list(phrases.export_phrases()))}")

corpus = df['nltk_lemmas'].values.tolist()
phrases = Phrases(corpus, min_count = 25, threshold = 40)
corpus = [token for token in phrases[corpus]]
df['nltk_phrases'] = corpus
ngrams = list(phrases.export_phrases())
print(ngrams)
print(len(ngrams))



We have obtained a large number of very interesting bigrams.
Visualize the distribution of token count per review (document).

In [ ]:
# Get the number of tokens per document
token_counts = [len(doc) for doc in corpus]

# Compute the average number of tokens per review
avg_tokens = np.mean(token_counts)

# Define x-axis limits (Modify these values as needed)
x_min = 0
x_max = max(token_counts) + 10
x_max_man = 1200

plt.figure(figsize=(8, 5))
plt.hist(token_counts, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel("Number of Tokens per Review")
plt.ylabel("Frequency")
plt.title("Histogram of Token Counts per Review")
plt.axvline(avg_tokens, color='red', linestyle='dashed', linewidth=2, label=f'Avg: {avg_tokens:.2f}')
plt.legend()
plt.xlim(x_min, x_max_man)

print(f"Average number of tokens per review: {avg_tokens:.2f}")

plt.show()
# Create a Gensim dictionary with the updatd `corpus`.
from gensim.corpora import Dictionary

D = Dictionary(corpus)
D.filter_extremes(no_below = 20, no_above = 0.80)
vocab = list(D.token2id.keys())
print(len(vocab))
# Save corpus as text file
with open("linkedin_corpus.txt", 'w', encoding='utf-8') as fout:
  for element in corpus:
    fout.write(' '.join(element) + '\n')

D.save("linkedin_dictionary.dict")
filepath = 'linkedin_corpus.txt'

with open(filepath, 'r') as f:
    corpus = f.read()


from gensim.corpora import Dictionary

class IterableCorpus_fromfile:
    def __init__(self, filename):
        self.__filename = filename

    def __iter__(self):
        with open(self.__filename, 'r', encoding='utf-8') as fin:
            for line in fin:
                yield line.strip().split()

MyIterCorpus = IterableCorpus_fromfile('linkedin_corpus.txt')
D = Dictionary.load("linkedin_dictionary.dict")

n_tokens = len(D)

##### FastText

In [ ]:
from gensim.models import FastText

model_fasttext = FastText(sentences = MyIterCorpus, vector_size = 300, window = 5, min_count = 20, sg = 1, seed = 42, workers = 4, epochs = 5)
# Save the word vectors of the FastText model.
from gensim.models import KeyedVectors
import gc

fasttext_wv = model_fasttext.wv
fasttext_wv.save("model_fastText.wordvectors")

del model_fasttext
gc.collect()
# Load previously saved word vectors of the FastText model.
from gensim.models import KeyedVectors

fastText_wv = KeyedVectors.load("model_fastText.wordvectors", mmap='r')
# Check results
print('============= FastText vocabulary =============')
words = list(fasttext_wv.key_to_index)
print(len(words))
print(words[99:130])
# Visualize the most relevant word vectors of the FastText model using TSNE.
from sklearn.manifold import TSNE

# Select the more relevant terms of the FastText vocabulary
words_ft = list(fastText_wv.index_to_key)[:2500]
word_vectors_ft = np.array([fastText_wv[word] for word in words])

# Run t-SNE
# Perplexity: parameter that allows
tsne_ft = TSNE(n_components=2, perplexity=50, random_state=42)
reduced_ft = tsne_ft.fit_transform(word_vectors_ft)

# Plot
plt.figure(figsize=(16, 12))
plt.scatter(reduced_ft[:, 0], reduced_ft[:, 1], alpha=0.7, s=10)

# Label only some words to avoid cluttering
for i, word in enumerate(words_ft[:180]):
    plt.text(reduced_ft[i, 0], reduced_ft[i, 1], word, fontsize=9)

plt.title("t-SNE visualization of FastText embeddings")
plt.show()


Obtain the same stats using the word vectors of the FastText model.

In [ ]:

from scipy.sparse import csr_matrix

def get_linkedin_vector(model, linkedin_description):
    word_vecs = []
    for token in linkedin_description:
        if token in model.key_to_index:
            word_vecs.append(model[token])
    if len(word_vecs) == 0:
        return np.zeros(model.vector_size)
    else:
        vec = np.mean(word_vecs, axis = 0)
    return vec
def get_vocabulary_coverage(model, gensim_dict):
    vocab = list(gensim_dict.token2id.keys())
    unknown_words = sorted(list(set(vocab).difference(set(model.key_to_index))))
    unknown_ids = [gensim_dict.token2id[w] for w in unknown_words]
    unknown_count = np.sum([gensim_dict.cfs[idx] for idx in unknown_ids])
    coverage = 1 - unknown_count / gensim_dict.num_pos
    return coverage
fT_coverage = get_vocabulary_coverage(fastText_wv, D)
print("Coverage {0:.4f}".format(fT_coverage))

linkedin_fT = np.array(
		df['nltk_lemmas'].swifter.apply(lambda r: get_linkedin_vector(fastText_wv, r)).tolist()
)
_fT_csr = csr_matrix(linkedin_fT)
from sklearn.manifold import TSNE

# Select the more relevant terms of the FastText vocabulary
words_ft = list(fastText_wv.index_to_key)[:2500]
word_vectors_ft = np.array([fastText_wv[word] for word in words])

# Run t-SNE
# Perplexity: parameter that allows
tsne_ft = TSNE(n_components=2, perplexity=50, random_state=42)
reduced_ft = tsne_ft.fit_transform(word_vectors_ft)

# Plot
plt.figure(figsize=(16, 12))
plt.scatter(reduced_ft[:, 0], reduced_ft[:, 1], alpha=0.7, s=10)

# Label only some words to avoid cluttering
for i, word in enumerate(words_ft[:180]):
    plt.text(reduced_ft[i, 0], reduced_ft[i, 1], word, fontsize=9)

plt.title("t-SNE visualization of FastText embeddings")
plt.show()

##### Pretrained FastText

In [ ]:

import compress_fasttext

fastTextPre = compress_fasttext.models.CompressedFastTextKeyedVectors.load(
    'https://github.com/avidale/compress-fasttext/releases/download/v0.0.4/cc.en.300.compressed.bin'
)
# Check results
print('============= pre-trained FastText vocabulary =============')
words = list(fastTextPre.key_to_index.keys())
print(len(words))
print(words[99:130])
fTpre_coverage = get_vocabulary_coverage(fastTextPre, D)
print("Coverage {0:.4f}".format(fT_coverage))

linkedin_fTpre = np.array(
		df['nltk_lemmas'].swifter.apply(lambda r: get_linkedin_vector(fastTextPre, r)).tolist()
)
linkedin_fTpre_csr = csr_matrix(linkedin_fTpre)
from sklearn.manifold import TSNE

# Select the more relevant terms of the FastText vocabulary
words_ft = list(fastTextPre.index_to_key)[:2500]
word_vectors_ft = np.array([fastTextPre[word] for word in words])

# Run t-SNE
# Perplexity: parameter that allows
tsne_ft = TSNE(n_components=2, perplexity=50, random_state=42)
reduced_ft = tsne_ft.fit_transform(word_vectors_ft)

# Plot
plt.figure(figsize=(16, 12))
plt.scatter(reduced_ft[:, 0], reduced_ft[:, 1], alpha=0.7, s=10)

# Label only some words to avoid cluttering
for i, word in enumerate(words_ft[:180]):
    plt.text(reduced_ft[i, 0], reduced_ft[i, 1], word, fontsize=9)

plt.title("t-SNE visualization of FastText embeddings")
plt.show()
from sklearn.manifold import TSNE

# Select the more relevant terms of the FastText vocabulary
words_ft = list(fastTextPre.index_to_key)[:2000]
word_vectors_ft = np.array([fastTextPre[word] for word in words])

# Run t-SNE
# Perplexity: parameter that allows
tsne_ft = TSNE(n_components=2, perplexity=50, random_state=42)
reduced_ft = tsne_ft.fit_transform(word_vectors_ft)

# Plot
plt.figure(figsize=(16, 12))
plt.scatter(reduced_ft[:, 0], reduced_ft[:, 1], alpha=0.7, s=10)

# Label only some words to avoid cluttering
for i, word in enumerate(words_ft[:180]):
    plt.text(reduced_ft[i, 0], reduced_ft[i, 1], word, fontsize=9)

plt.title("t-SNE visualization of FastText embeddings")
plt.show()





##### Selected model for description

As the T-SNE has given better results in the Fast Text that the one is going to be used

In [ ]:
def sentence_to_vector(sentence):
    words = sentence.split()
    vectors = [fastText_wv[w] for w in words if w in fastText_wv]

    if len(vectors) == 0:
        return np.zeros(fastText_wv.vector_size)

    return np.mean(vectors, axis=0)

df['description'] = df['description'].apply(sentence_to_vector)

print(df['description'].iloc[0])        # vector of first row
print(df['description'].iloc[0].shape)  # shape (dimension,)

df.to_csv("linkedin_description_vectors.csv", index=False)

#### Load data with vectors as 'description'

In [7]:
# To reload
file_name= "linkedin_description_vectors.csv"
df=pd.read_csv(file_name)
print(f"Data has been recovered from csv file.")

Data has been recovered from csv file.


In [8]:
# Take a first look into data
print("====== Number of samples ======")
print("Number of samples: ", len(df))
print("\n====== Check NAs ======")
print(df.isna().sum().to_frame(name="NA_Counts"))

====== Number of samples ======
Number of samples:  10490

====== Check NAs ======
                   NA_Counts
offer_id                   0
title                      0
company_name               0
city                     408
state_code                 0
seniority_level            0
job_type                   0
job_function               5
industry                   1
description                0
salary_min              3673
salary_max              3673
salary_unit             3673
salary                  3673
searched_position          0
searched_location          0
applications               0
nltk_lemmas                0


In [9]:
print("Number of samples by job_type: ")
print(df.groupby("job_type").size().reset_index(name = "count"))

Number of samples by job_type: 
     job_type  count
0    Contract    634
1   Full-time   9508
2  Internship    220
3       Other     18
4   Part-time     72
5   Temporary     38


In [10]:
# Eliminate volunteer positions (only 6)
df = df[(df['job_type']!='Volunteer')]

In [ ]:
# To save with the volunteer positions eliminated
file_name= "linkedin_description_vectors.csv"
df.to_csv(file_name, index=False)
print(f"Data has been saved to file: {file_name}")

#### Salary conversion to $/year

In [22]:
# To reload after volunteer elimination and with embeddings in description
file_name= "linkedin_description_vectors.csv"
df=pd.read_csv(file_name)
print(f"Data has been recovered from csv file.")

Data has been recovered from csv file.


In [23]:
print("\n====== Check NAs ======")
print(df.isna().sum().to_frame(name="NA_Counts"))


====== Check NAs ======
                   NA_Counts
offer_id                   0
title                      0
company_name               0
city                     408
state_code                 0
seniority_level            0
job_type                   0
job_function               5
industry                   1
description                0
salary_min              3673
salary_max              3673
salary_unit             3673
salary                  3673
searched_position          0
searched_location          0
applications               0
nltk_lemmas                0


In [12]:
df["salary_unit"].unique()

array(['year', nan, 'hour', 'month'], dtype=object)

In [24]:
from importlib import reload
import utilities
reload(utilities)

<module 'utilities' from 'C:\\Users\\adeli\\Documents 4-Q1\\Web Analytics\\Final Project GitHub\\web-analytics-project-repo\\utilities.py'>

In [25]:
from utilities import salary_conversion

In [26]:
df["job_type"].value_counts()

job_type
Full-time     9508
Contract       634
Internship     220
Part-time       72
Temporary       38
Other           18
Name: count, dtype: int64

In [27]:
# Need to map each job_type to hours_a_week, modify if needed
hours_map = {
    "Full-time": 40,
    "Part-time": 20,
    "Internship": 40,
    "Contract": 40,
    "Temporary": 40,
    "Other": 40,
}

# Create hours_a_week column
df["hours_a_week"] = df["job_type"].map(hours_map)

In [28]:
def convert_row_to_year(row):
    min_sal = row["salary_min"]
    max_sal = row["salary_max"]
    unit_in = row["salary_unit"]      # 'year', nan, 'hour', 'month'
    hours   = row["hours_a_week"]     # take from the mapping defined above

    # If any of these are missing, keep NaNs
    if pd.isna(min_sal) or pd.isna(max_sal) or pd.isna(unit_in):
        return pd.Series(
            [np.nan, np.nan],
            index=["salary_min_year", "salary_max_year"]
        )

    # Use the salary conversion function (utilities.py)
    min_y, max_y = salary_conversion(min_sal, max_sal, unit_in, "year", hours)
    return pd.Series(
        [min_y, max_y],
        index=["salary_min_year", "salary_max_year"]
    )


In [29]:
df[["salary_min_year", "salary_max_year"]] = df.apply(convert_row_to_year, axis=1)

In [31]:
df["salary_avg_year"] = df[["salary_min_year", "salary_max_year"]].mean(axis=1)

In [ ]:
# df.drop(columns=["nltk_lemmas"]).head(1) 

In [32]:
print("\n====== Check NAs ======")
print(df.isna().sum().to_frame(name="NA_Counts"))


====== Check NAs ======
                   NA_Counts
offer_id                   0
title                      0
company_name               0
city                     408
state_code                 0
seniority_level            0
job_type                   0
job_function               5
industry                   1
description                0
salary_min              3673
salary_max              3673
salary_unit             3673
salary                  3673
searched_position          0
searched_location          0
applications               0
nltk_lemmas                0
hours_a_week               0
salary_min_year         3673
salary_max_year         3673
salary_avg_year         3673


In [33]:
# To save
file_name= "linkedin_converted_salaries_year.csv"
df.to_csv(file_name, index=False)
print(f"Data has been saved to file: {file_name}")

Data has been saved to file: linkedin_converted_salaries_year.csv


#### Recover data from linkedin_converted_salaries.csv

In [10]:
file_name= "linkedin_converted_salaries.csv"
df=pd.read_csv(file_name)
print(f"Data has been recovered from csv file.")

Data has been recovered from csv file.


In [12]:
df.columns

Index(['offer_id', 'title', 'company_name', 'city', 'state_code',
       'seniority_level', 'job_type', 'job_function', 'industry',
       'description', 'salary_min', 'salary_max', 'salary_unit', 'salary',
       'searched_position', 'searched_location', 'applications', 'nltk_lemmas',
       'hours_a_week', 'salary_min_year', 'salary_max_year'],
      dtype='object')

In [1]:
# df.head(1)

### Transform dataset to salary per hour

In [8]:
# To reload after volunteer elimination and with embeddings in description
file_name= "linkedin_description_vectors.csv"
df=pd.read_csv(file_name)
print(f"Data has been recovered from csv file.")

Data has been recovered from csv file.


In [9]:
print("\n====== Check NAs ======")
print(df.isna().sum().to_frame(name="NA_Counts"))


====== Check NAs ======
                   NA_Counts
offer_id                   0
title                      0
company_name               0
city                     408
state_code                 0
seniority_level            0
job_type                   0
job_function               5
industry                   1
description                0
salary_min              3673
salary_max              3673
salary_unit             3673
salary                  3673
searched_position          0
searched_location          0
applications               0
nltk_lemmas                0


In [12]:
from importlib import reload
import utilities
reload(utilities)

<module 'utilities' from 'C:\\Users\\adeli\\Documents 4-Q1\\Web Analytics\\Final Project GitHub\\web-analytics-project-repo\\utilities.py'>

In [13]:
from utilities import salary_conversion

In [14]:
# Need to map each job_type to hours_a_week, modify if needed
hours_map = {
    "Full-time": 40,
    "Part-time": 20,
    "Internship": 40,
    "Contract": 40,
    "Temporary": 40,
    "Other": 40,
}

# Create hours_a_week column
df["hours_a_week"] = df["job_type"].map(hours_map)

In [15]:
def convert_row_to_hour(row):
    min_sal = row["salary_min"]
    max_sal = row["salary_max"]
    unit_in = row["salary_unit"]      # 'year', nan, 'hour', 'month'
    hours   = row["hours_a_week"]     # take from the mapping defined above
    
    # If any of these are missing, keep NaNs
    if pd.isna(min_sal) or pd.isna(max_sal) or pd.isna(unit_in):
        return pd.Series(
            [np.nan, np.nan],
            index=["salary_min_hour", "salary_max_hour"]
        )

    # Use the salary conversion function (utilities.py)
    min_y, max_y = salary_conversion(min_sal, max_sal, unit_in, "hour", hours)
    return pd.Series(
        [min_y, max_y],
        index=["salary_min_hour", "salary_max_hour"]
    )

In [16]:
# Apply the function to create the new columns, and the average salary per hour
df[["salary_min_hour", "salary_max_hour"]] = df.apply(convert_row_to_hour, axis=1)
df["salary_avg_hour"] = df[["salary_min_hour", "salary_max_hour"]].mean(axis=1)


In [19]:
df.columns

Index(['offer_id', 'title', 'company_name', 'city', 'state_code',
       'seniority_level', 'job_type', 'job_function', 'industry',
       'description', 'salary_min', 'salary_max', 'salary_unit', 'salary',
       'searched_position', 'searched_location', 'applications', 'nltk_lemmas',
       'hours_a_week', 'salary_min_hour', 'salary_max_hour',
       'salary_avg_hour'],
      dtype='object')

In [21]:
print("\n====== Check NAs ======")
print(df.isna().sum().to_frame(name="NA_Counts"))


====== Check NAs ======
                   NA_Counts
offer_id                   0
title                      0
company_name               0
city                     408
state_code                 0
seniority_level            0
job_type                   0
job_function               5
industry                   1
description                0
salary_min              3673
salary_max              3673
salary_unit             3673
salary                  3673
searched_position          0
searched_location          0
applications               0
nltk_lemmas                0
hours_a_week               0
salary_min_hour         3673
salary_max_hour         3673
salary_avg_hour         3673


In [ ]:
# To save
file_name= "linkedin_converted_salaries_hour.csv"
df.to_csv(file_name, index=False)
print(f"Data has been saved to file: {file_name}")
# df.to_csv("linkedin_converted_salaries_hour.csv", index=False)

#### Legacy code

In [ ]:
# Used to correct three entries that were outliers because they had salary_unit as day
# when the should have been year
ids_with_nan = [115, 1350, 6894] 
df.loc[df["offer_id"].isin(ids_with_nan), "salary_unit"] = "year"